# Train/Test Split & Data Leakage
**Day 49 — Phase 4: Machine Learning**

## Objective
- Understand why we split data into train and test sets.
- Use `train_test_split` from `sklearn.model_selection` correctly.
- Understand data leakage, and why splitting must happen BEFORE preprocessing.
- See the difference between scaling before split (wrong) vs after split (correct).

## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Theory
- **Train/test split:** hold out part of the data so the model is evaluated on data it never trained on.
- **Data leakage:** information from outside training (often the test set) sneaks into the training process, inflating evaluation scores.
- **Golden rule:** split first, preprocess second — always fit any transformer (scaler, encoder) on the training set only.

## Example 1 — Basic train_test_split on FC Lahore Lions match data

In [ ]:
# Small synthetic FC Lahore Lions match dataset
data = pd.DataFrame({
    "shots_on_target": [8, 3, 6, 2, 9, 4, 7, 1, 5, 6],
    "possession_pct":  [61, 42, 55, 38, 66, 47, 58, 33, 50, 53],
    "corners":         [6, 2, 4, 1, 7, 3, 5, 0, 3, 4],
    "result":          [1, 0, 1, 0, 1, 0, 1, 0, 1, 1]  # 1 = win, 0 = not win
})

X = data.drop(columns=["result"])
y = data["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train), "| Test size:", len(X_test))

## Example 2 — Data leakage: scaling BEFORE split (wrong) vs AFTER split (correct)

In [ ]:
# WRONG: scale the whole dataset first, then split
scaler_leaky = StandardScaler()
X_scaled_leaky = scaler_leaky.fit_transform(X)  # sees ALL rows, including future test rows

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_scaled_leaky, y, test_size=0.2, random_state=42
)

model_leaky = LogisticRegression().fit(X_train_leaky, y_train_leaky)
leaky_score = accuracy_score(y_test_leaky, model_leaky.predict(X_test_leaky))
print("Leaky pipeline test accuracy:", leaky_score)

In [ ]:
# CORRECT: split first, THEN fit the scaler on train only
scaler_clean = StandardScaler()
X_train_scaled = scaler_clean.fit_transform(X_train)          # fit on train only
X_test_scaled = scaler_clean.transform(X_test)                 # transform test using train's stats

model_clean = LogisticRegression().fit(X_train_scaled, y_train)
clean_score = accuracy_score(y_test, model_clean.predict(X_test_scaled))
print("Leakage-free pipeline test accuracy:", clean_score)

## Practice Exercise
- Change `test_size` to `0.3` and re-run. How do train/test sizes change?
- Change `random_state` to a different number. Does the split change? Why?

In [ ]:
# Your turn: experiment with test_size and random_state


## Mini Challenge
- On this tiny dataset the leaky vs. clean scores may look similar (too few rows to show the gap clearly).
- Explain in your own words, in a markdown cell, why the leaky version is still wrong even if the numbers happen to match here.

*(write your explanation here)*

## Summary
- Always split data into train/test BEFORE any preprocessing.
- `train_test_split(X, y, test_size=0.2, random_state=42)` is the standard, reproducible pattern.
- Data leakage happens when test-set information (even indirectly, via a scaler fit on the whole dataset) influences training.
- Fit transformers (scalers, encoders) on the training set only, then apply them to both sets.
- Next: building the first real supervised model on this clean, correctly-split data.